In [9]:
from dataclasses import dataclass, field

from dowhy import CausalModel
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
NUM_SIMULATIONS = 100

In [3]:
@dataclass
class ModelResult:
    treatment: any
    data: any
    outcome: str
    confounders: list
    method_params: dict

    model: any = field(init=False)
    estimand: any = field(init=False)
    estimate: any = field(init=False)

    def fit(self):
        self.model = CausalModel(data=self.data,
                                 treatment=self.treatment,
                                 outcome=self.outcome,
                                 common_causes=self.confounders)

        self.estimand = self.model.identify_effect()

        self.estimate = self.model.estimate_effect(
            identified_estimand=self.estimand,
            method_name="backdoor.econml.dml.LinearDML",
            method_params=self.method_params)

    def run_refuters(self):
        results = []

        refuters = [("data_subset", {
            "method_name": "data_subset_refuter",
            "subset_fraction": 0.8,
            "num_simulations": NUM_SIMULATIONS
        }),
                    ("random_common_cause", {
                        "method_name": "random_common_cause",
                        "num_simulations": NUM_SIMULATIONS
                    }),
                    ("random_treatment", {
                        "method_name": "placebo_treatment_refuter",
                        "num_simulations": NUM_SIMULATIONS
                    }),
                    ("random_outcome", {
                        "method_name": "dummy_outcome_refuter",
                        "num_simulations": NUM_SIMULATIONS
                    })]

        for name, params in refuters:
            refute = self.model.refute_estimate(self.estimand, self.estimate,
                                                **params)

            refute_res = refute[0] if isinstance(refute, list) else refute
            results.append({
                "treatment":
                self.treatment,
                "refuter":
                name,
                "original_effect":
                refute_res.estimated_effect,
                "new_effect":
                refute_res.new_effect,
                "delta":
                refute_res.new_effect - refute_res.estimated_effect,
                "p_value":
                refute_res.refutation_result.get("p_value", None)
            })

        return results

In [4]:
df = pd.read_csv('../data/software_usage_promotion.csv')

df['Both'] = (df['Discount'] *
                            df['Tech Support']).astype('category')
confounders = [
    'Size', 'Employee Count', 'PC Count', 'IT Spend', 'Major Flag',
    'Global Flag', 'Commercial Flag', 'SMC Flag'
]

In [10]:
tree_model = DecisionTreeClassifier(max_depth=3)

model_t = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression())
])

In [12]:
model_obj = {}
model_results = {}

for i in ['Discount', 'Tech Support', 'Both']:
    model = ModelResult(treatment=i,
                        data=df,
                        outcome='Revenue',
                        confounders=confounders,
                        method_params={
                            'init_params': {
                                'model_y': GradientBoostingRegressor(),
                                #'model_t': tree_model,
                                'model_t': model_t,
                                'discrete_treatment': True
                            },
                            'fit_params': {}
                        })
    model.fit()
    model_obj[i] = model
    res = model.run_refuters()
    model_results[i] = res

In [13]:
model_results['Discount']

[{'treatment': 'Discount',
  'refuter': 'data_subset',
  'original_effect': np.float64(5020.831288112654),
  'new_effect': np.float64(5039.663585050612),
  'delta': np.float64(18.832296937957835),
  'p_value': np.float64(0.8999999999999999)},
 {'treatment': 'Discount',
  'refuter': 'random_common_cause',
  'original_effect': np.float64(5020.831288112654),
  'new_effect': np.float64(5041.430338510906),
  'delta': np.float64(20.599050398252075),
  'p_value': np.float64(0.72)},
 {'treatment': 'Discount',
  'refuter': 'random_treatment',
  'original_effect': np.float64(5020.831288112654),
  'new_effect': np.float64(10.962305948658457),
  'delta': np.float64(-5009.8689821639955),
  'p_value': np.float64(0.8999999999999999)},
 {'treatment': 'Discount',
  'refuter': 'random_outcome',
  'original_effect': 0,
  'new_effect': np.float64(0.0066132068652458706),
  'delta': np.float64(0.0066132068652458706),
  'p_value': np.float64(0.8600000000000001)}]

In [14]:
model_results['Tech Support']

[{'treatment': 'Tech Support',
  'refuter': 'data_subset',
  'original_effect': np.float64(6930.32119367849),
  'new_effect': np.float64(6893.703488897221),
  'delta': np.float64(-36.61770478126891),
  'p_value': np.float64(0.78)},
 {'treatment': 'Tech Support',
  'refuter': 'random_common_cause',
  'original_effect': np.float64(6930.32119367849),
  'new_effect': np.float64(6881.836964714303),
  'delta': np.float64(-48.48422896418742),
  'p_value': np.float64(0.52)},
 {'treatment': 'Tech Support',
  'refuter': 'random_treatment',
  'original_effect': np.float64(6930.32119367849),
  'new_effect': np.float64(18.748793913139007),
  'delta': np.float64(-6911.572399765351),
  'p_value': np.float64(0.98)},
 {'treatment': 'Tech Support',
  'refuter': 'random_outcome',
  'original_effect': 0,
  'new_effect': np.float64(0.007098381422382883),
  'delta': np.float64(0.007098381422382883),
  'p_value': np.float64(0.96)}]

In [15]:
model_results['Both']

[{'treatment': 'Both',
  'refuter': 'data_subset',
  'original_effect': np.float64(8806.782795673413),
  'new_effect': np.float64(8832.472617932406),
  'delta': np.float64(25.689822258993445),
  'p_value': np.float64(0.8600000000000001)},
 {'treatment': 'Both',
  'refuter': 'random_common_cause',
  'original_effect': np.float64(8806.782795673413),
  'new_effect': np.float64(8821.626213949452),
  'delta': np.float64(14.843418276039301),
  'p_value': np.float64(0.8799999999999999)},
 {'treatment': 'Both',
  'refuter': 'random_treatment',
  'original_effect': np.float64(8806.782795673413),
  'new_effect': np.float64(21.635681589615796),
  'delta': np.float64(-8785.147114083797),
  'p_value': np.float64(0.8400000000000001)},
 {'treatment': 'Both',
  'refuter': 'random_outcome',
  'original_effect': 0,
  'new_effect': np.float64(-0.0033410747941632424),
  'delta': np.float64(-0.0033410747941632424),
  'p_value': np.float64(0.96)}]

All refuters pass.

Note that when we used a logistic regression with raw variables, the modeling did not converge.